# 10 · Hierarchical (Coherent) Probabilistic Forecasts

```
                 Total
          ┌────────┴────────┐
        North             South
      ┌───┴───┐         ┌───┴───┐
   North_A North_B   South_A South_B
```
Series that **aggregate** (product → category → total, store → region → country, monthly → quarterly) must have
**coherent** forecasts: the Total forecast should equal the sum of its children. Independently produced base
forecasts are almost never coherent.

## Theory

### Notation
Bottom series $\mathbf b_t\in\mathbb R^m$ ($m=4$), all series $\mathbf y_t=\mathbf S\mathbf b_t\in\mathbb R^n$ ($n=7$), where
$\mathbf S$ is the **summing matrix**. A forecast is coherent if it lies in the *coherent subspace* spanned by the columns of $\mathbf S$.

### Point reconciliation: $\tilde{\mathbf y}=\mathbf S\mathbf G\hat{\mathbf y}$
| Method | $\mathbf G$ |
|---|---|
| Bottom-up | $[\mathbf 0\ \ \mathbf I_m]$ (use only the bottom forecasts) |
| OLS | $(\mathbf S^\top\mathbf S)^{-1}\mathbf S^\top$ |
| WLS (structural) | $W=\text{diag}(\mathbf S\mathbf 1)$, i.e. weights ∝ number of bottom series |
| **MinT** (Wickramasuriya, Athanasopoulos & Hyndman 2019) | $(\mathbf S^\top\mathbf W^{-1}\mathbf S)^{-1}\mathbf S^\top\mathbf W^{-1}$ with $\mathbf W$ = covariance of base forecast errors (shrinkage estimate) |

MinT minimises the trace of the reconciled error covariance, $\mathbf S\mathbf G\mathbf W\mathbf G^\top\mathbf S^\top$.
It uses information from **all levels**. The aggregate is usually easier to forecast, and the bottom carries the detail.

### Probabilistic reconciliation (Panagiotelis, Gamakumara, Athanasopoulos & Hyndman 2023)
A coherent *distribution* puts all its mass on the coherent subspace. Practical recipe:
1. Draw **joint** samples $\hat{\mathbf y}^{(k)}$ from the base forecasts. They must keep the dependence *between* series
   (copula / joint bootstrap of residual vectors, nb 09).
2. Project every sample: $\tilde{\mathbf y}^{(k)}=\mathbf S\mathbf G\hat{\mathbf y}^{(k)}$.

For Gaussian base forecasts $N(\hat{\mathbf y},\boldsymbol\Sigma)$ the reconciled distribution is
$N(\mathbf S\mathbf G\hat{\mathbf y},\ \mathbf S\mathbf G\boldsymbol\Sigma\mathbf G^\top\mathbf S^\top)$.

**Classic mistake:** bottom-up with *independent* bottom-level samples gives a far too narrow Total distribution
(nb 09: dependence matters for sums).

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import plotly.graph_objects as go
from scipy import stats
from statsmodels.tsa.statespace.sarimax import SARIMAX
import pf_utils as pf
pf.set_style()
rng = np.random.default_rng(10)

retail = pf.make_retail_hierarchy()
names = pf.HIER_NAMES
S = pf.summing_matrix()
LEVEL = {"Total": ["Total"], "Region": ["North", "South"], "Store": pf.BOTTOM_NAMES}
print("S =\n", S.astype(int))
print("data coherent:", np.allclose(retail.to_numpy(), retail[pf.BOTTOM_NAMES].to_numpy() @ S.T))

S =
 [[1 1 1 1]
 [1 1 0 0]
 [0 0 1 1]
 [1 0 0 0]
 [0 1 0 0]
 [0 0 1 0]
 [0 0 0 1]]
data coherent: True


## 1. Base forecasts for all 7 series (independently, one-step ahead)
Each series gets its own SARIMA on log-sales (parameters from 2015–2019). We forecast 2020–2024 one step ahead. Base
point forecast = predictive median $e^{\mu}$.

In [2]:
CUT, START = 60, 13
MU, SD, Z, E = [], [], [], []
for s in names:
    ly = np.log(retail[s])
    fit = SARIMAX(ly.iloc[:CUT], order=(0, 1, 1), seasonal_order=(0, 1, 1, 12)).fit(disp=False)
    pr = SARIMAX(ly, order=(0, 1, 1), seasonal_order=(0, 1, 1, 12)).filter(fit.params).get_prediction(start=START)
    m, v = np.asarray(pr.predicted_mean), np.sqrt(np.asarray(pr.var_pred_mean))
    k = CUT - START
    Z.append(((ly.iloc[START:].to_numpy() - m) / v)[:k])                 # standardised log residuals (train)
    E.append((retail[s].iloc[START:].to_numpy() - np.exp(m))[:k])         # level-scale residuals (train) for W
    MU.append(m[k:]); SD.append(v[k:])
MU, SD, Z, E = (np.column_stack(a) for a in (MU, SD, Z, E))
Y = retail.iloc[CUT:].to_numpy()
T = len(Y)
base_point = np.exp(MU)
incoh = base_point[:, 0] - base_point[:, 3:].sum(axis=1)
print(f"Base forecasts: Total minus sum of stores, mean |gap| = {np.abs(incoh).mean():.1f}, max = {np.abs(incoh).max():.1f}")

Base forecasts: Total minus sum of stores, mean |gap| = 20.2, max = 66.6


## 2. Reconciliation matrices

In [3]:
def G_from_W(W):
    Wi = np.linalg.inv(W)
    return np.linalg.solve(S.T @ Wi @ S, S.T @ Wi)

def shrink_cov(e):
    """Schäfer–Strimmer shrinkage towards the diagonal (as in the R `hts`/`fable` MinT-shrink)."""
    n = len(e)
    covm = e.T @ e / n
    tar = np.diag(np.diag(covm))
    xs = e / np.sqrt(np.diag(covm))
    v = (1 / (n * (n - 1))) * ((xs**2).T @ (xs**2) - (xs.T @ xs) ** 2 / n)
    np.fill_diagonal(v, 0)
    corm = covm / np.sqrt(np.outer(np.diag(covm), np.diag(covm)))
    d = (corm - np.eye(len(corm))) ** 2
    lam = float(np.clip(v.sum() / d.sum(), 0, 1))
    return lam * tar + (1 - lam) * covm, lam

W_shr, lam = shrink_cov(E)
print(f"shrinkage intensity λ = {lam:.2f}")
G = {
    "Bottom-up": np.hstack([np.zeros((4, 3)), np.eye(4)]),
    "OLS": G_from_W(np.eye(7)),
    "WLS (structural)": G_from_W(np.diag(S.sum(axis=1))),
    "MinT-shrink": G_from_W(W_shr),
}
pd.DataFrame(S @ G["MinT-shrink"], index=names, columns=names).round(2)

shrinkage intensity λ = 0.07


,Total,North,South,North_A,North_B,South_A,South_B
Total,0.26,0.31,0.18,0.44,0.44,0.56,0.56
North,0.08,0.31,-0.13,0.61,0.61,0.05,0.05
South,0.18,-0.00,0.31,-0.18,-0.18,0.51,0.51
North_A,0.15,0.18,-0.11,0.68,-0.32,-0.03,-0.03
North_B,-0.07,0.13,-0.02,-0.07,0.93,0.08,0.08
South_A,0.09,0.01,0.20,-0.11,-0.11,0.71,-0.29
South_B,0.09,-0.02,0.12,-0.07,-0.07,-0.20,0.80


The rows of $\mathbf{SG}$ show how much each reconciled series borrows from every base forecast. For example,
MinT's reconciled Total is a weighted mix of the base Total, the base regions and the base stores.

## 3. Point reconciliation

In [4]:
pt = {"Base (incoherent)": base_point}
for k, g in G.items():
    pt[k] = base_point @ (S @ g).T
rmse = {k: {lvl: np.sqrt(np.mean((v[:, [names.index(c) for c in cols]] - Y[:, [names.index(c) for c in cols]]) ** 2))
            for lvl, cols in LEVEL.items()} for k, v in pt.items()}
pd.DataFrame(rmse).T.round(1)

,Total,Region,Store
Base (incoherent),114.9,69.2,38.1
Bottom-up,114.0,69.1,38.1
OLS,114.0,69.0,38.1
WLS (structural),113.8,68.9,38.0
MinT-shrink,113.7,68.9,38.1


## 4. Probabilistic reconciliation
Joint base samples via a Gaussian copula on the log scale with the 7×7 residual correlation (nb 09), then project every
sample with $\mathbf{SG}$. For comparison we also include **bottom-up from independent store samples**.

In [5]:
M = 500
R7 = np.corrcoef(Z.T) + 1e-6 * np.eye(7)
L7 = np.linalg.cholesky(R7)
base_smp = np.exp(MU[:, None, :] + SD[:, None, :] * (rng.standard_normal((T, M, 7)) @ L7.T))     # (T, M, 7)
indep_bottom = np.exp(MU[:, None, 3:] + SD[:, None, 3:] * rng.standard_normal((T, M, 4)))

P = {"Base (incoherent)": base_smp,
     "Bottom-up, independent stores ✗": indep_bottom @ S.T}
for k, g in G.items():
    P[k] = base_smp @ (S @ g).T

scale = retail.iloc[:CUT].std().to_numpy()             # put series on comparable scales for the energy score
rows = {}
for k, smp in P.items():
    r = {}
    for lvl, cols in LEVEL.items():
        idx = [names.index(c) for c in cols]
        r[f"CRPS {lvl}"] = np.mean([pf.crps_sample(Y[:, i], smp[:, :, i].T).mean() for i in idx])
    tot = smp[:, :, 0]
    r["90% cov. Total"] = pf.coverage(Y[:, 0], np.quantile(tot, .05, axis=1), np.quantile(tot, .95, axis=1))
    r["energy (scaled)"] = np.mean([pf.energy_score(Y[t] / scale, smp[t] / scale) for t in range(T)])
    r["max incoherence"] = np.abs(smp[:, :, 0] - smp[:, :, 3:].sum(axis=2)).max()
    rows[k] = r
pd.DataFrame(rows).T

,CRPS Total,CRPS Region,CRPS Store,90% cov. Total,energy (scaled),max incoherence
Base (incoherent),64.402,39.045,21.178,0.933,0.900,1.602e+02
"Bottom-up, independent stores ✗",64.560,38.966,21.066,0.800,0.904,0.000e+00
Bottom-up,63.809,38.987,21.178,0.900,0.897,0.000e+00
OLS,63.796,38.913,21.132,0.917,0.896,1.364e-12
WLS (structural),63.676,38.884,21.122,0.917,0.895,1.364e-12
MinT-shrink,63.623,38.851,21.137,0.917,0.896,9.095e-13


In [6]:
t0 = 36
SHOW = ["Base (incoherent)", "Bottom-up, independent stores ✗", "MinT-shrink"]
fig = go.Figure()
top = 0
for k, name in enumerate(SHOW):
    vals = P[name][t0, :, 0]
    top = max(top, np.histogram(vals, bins=40, density=True)[0].max())
    pf.add_step_hist(fig, vals, bins=40, name=name, color=pf.SERIES[k])
pf.add_vline(fig, Y[t0, 0], 0, top, "actual", color=pf.ACTUAL, dash="solid", width=2)
fig.update_layout(title=f"Predictive distribution of the TOTAL, {retail.index[CUT + t0]:%b %Y}", height=400,
                  xaxis_title="total sales", yaxis_title="density")
fig.show()

q = {tau: np.quantile(P["MinT-shrink"][:, :, 0], tau, axis=1) for tau in [0.05, 0.25, 0.5, 0.75, 0.95]}
fig = pf.add_fan(go.Figure(), retail.index[CUT:], q, history=retail["Total"].iloc[:CUT], actual=Y[:, 0], n_history=24,
                 fmt=",.0f", label="MinT")
fig.update_layout(title="MinT-reconciled one-step forecasts for the Total", yaxis_title="sales")
fig.show()

**Reading the results.** All reconciled forecasts are exactly coherent (incoherence ≈ 1e-12). The accuracy gains
are small here, because the synthetic data are generated bottom-up from smooth store series and every base model
is already good. Gains are typically much larger with noisy / intermittent bottom series, or when some level's
base model is misspecified. The *independent* bottom-up forecaster is coherent too, but its Total intervals
under-cover. Coherence alone does not mean correct dependence.

## 5. Takeaways
* Reconciliation makes forecasts **coherent** (incoherence becomes 0) and often **more accurate** than the base forecasts,
  because the projection pools information across levels. MinT is usually the best default.
* Probabilistic reconciliation = **joint base samples + projection**. The dependence between series must be in
  the samples. Otherwise the aggregate distribution is wrong, as with independent bottom-up.
* Evaluate at **every level** (CRPS per level) and jointly (energy / variogram score on scaled series).
* Python libraries: `hierarchicalforecast` (Nixtla) implements BottomUp, MinTrace, ERM and probabilistic
  methods (bootstrap, normality, PERMBU). In R: `fable` / `hts`.

### Exercises
1. Use `MinT-sample` (unshrunk covariance) and compare it with the shrinkage version. With 47 residual vectors and 7 series it is still invertible. Try with 15 vectors.
2. Add a temporal hierarchy: monthly → quarterly → annual for the Total (*temporal reconciliation*, Athanasopoulos et al. 2017).
3. Replace the Gaussian copula with joint bootstrapping of the 7-dimensional standardised residual vectors.